# 09 — Conceptual Diagrams (Phase 7)
## V31–V38: Architecture, Flowcharts, and Reference Diagrams

All diagrams generated programmatically with matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, ArrowStyle
import numpy as np
from pathlib import Path

FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Shared style
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.size': 10,
})

# Colour palette
C = {
    'blue': '#3498db', 'dark_blue': '#2c3e50', 'green': '#2ecc71',
    'red': '#e74c3c', 'orange': '#f39c12', 'purple': '#9b59b6',
    'teal': '#1abc9c', 'grey': '#95a5a6', 'light_grey': '#ecf0f1',
    'white': '#ffffff', 'light_blue': '#d6eaf8', 'light_green': '#d5f5e3',
    'light_red': '#fadbd8', 'light_orange': '#fdebd0', 'light_purple': '#e8daef',
}

def add_box(ax, x, y, w, h, text, color, text_color='white', fontsize=9,
            fontstyle='normal', fontweight='bold', boxstyle='round,pad=0.3',
            edgecolor='#2c3e50', linewidth=1.5):
    """Add a rounded box with centered text."""
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle=boxstyle, facecolor=color,
                          edgecolor=edgecolor, linewidth=linewidth)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight=fontweight, fontstyle=fontstyle,
            wrap=True)
    return box

def add_arrow(ax, x1, y1, x2, y2, color='#2c3e50', style='->', lw=1.5):
    """Add an arrow between two points."""
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle=style, color=color, lw=lw))

---
## V31 — System Architecture Diagram

End-to-end pipeline: Data → Preprocessing → FL Clients → Server Aggregation → Ensemble → SHAP → Prediction

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')

# Title
ax.text(8, 9.5, 'System Architecture', ha='center', va='center',
        fontsize=16, fontweight='bold', color=C['dark_blue'])
ax.text(8, 9.1, 'Explainable Federated Ensemble Learning for Financial Fraud Detection',
        ha='center', va='center', fontsize=10, color=C['grey'], fontstyle='italic')

# --- Row 1: Data Sources ---
y1 = 7.8
add_box(ax, 2.5, y1, 2.2, 0.8, 'ULB Dataset\n(284K txns)', C['blue'])
add_box(ax, 5.5, y1, 2.2, 0.8, 'BAF Dataset\n(1M txns)', C['blue'])
add_box(ax, 8.5, y1, 2.2, 0.8, 'Synthetic Dataset\n(5M txns)', C['blue'])
ax.text(12.5, y1, 'Data\nAcquisition', ha='center', va='center',
        fontsize=9, color=C['grey'], fontstyle='italic')

# --- Row 2: Preprocessing ---
y2 = 6.3
add_box(ax, 3, y2, 2.5, 0.8, 'Cleaning &\nEncoding', C['teal'])
add_box(ax, 6, y2, 2.5, 0.8, 'StandardScaler\nNormalization', C['teal'])
add_box(ax, 9, y2, 2.5, 0.8, 'SMOTE\nOversampling', C['teal'])
ax.text(12.5, y2, 'Preprocessing\nPipeline', ha='center', va='center',
        fontsize=9, color=C['grey'], fontstyle='italic')

# Arrows data -> preprocessing
for x in [2.5, 5.5, 8.5]:
    add_arrow(ax, x, y1 - 0.4, x + (5.5 - x) * 0.1, y2 + 0.4)

# Arrows within preprocessing
add_arrow(ax, 4.25, y2, 4.75, y2)
add_arrow(ax, 7.25, y2, 7.75, y2)

# --- Row 3: FL Clients ---
y3 = 4.5
# Background box for FL section
fl_bg = FancyBboxPatch((0.5, 3.0), 11, 2.2, boxstyle='round,pad=0.2',
                        facecolor=C['light_blue'], edgecolor=C['blue'],
                        linewidth=1.5, linestyle='--', alpha=0.4)
ax.add_patch(fl_bg)
ax.text(0.9, 5.0, 'Federated Learning (Flower)', fontsize=9,
        fontweight='bold', color=C['blue'])

add_box(ax, 2.5, y3, 2.2, 0.8, 'Client 1 (ULB)\nXGB + RF', C['purple'])
add_box(ax, 5.5, y3, 2.2, 0.8, 'Client 2 (BAF)\nXGB + RF', C['purple'])
add_box(ax, 8.5, y3, 2.2, 0.8, 'Client 3 (Synth)\nXGB + RF', C['purple'])

# Arrows preprocessing -> FL clients
add_arrow(ax, 6, y2 - 0.4, 2.5, y3 + 0.4, color=C['teal'])
add_arrow(ax, 6, y2 - 0.4, 5.5, y3 + 0.4, color=C['teal'])
add_arrow(ax, 6, y2 - 0.4, 8.5, y3 + 0.4, color=C['teal'])

# Server
add_box(ax, 13.5, y3, 2.2, 1.2, 'Flower Server\n\nFedAvg\nAggregation', C['dark_blue'])

# Bidirectional arrows clients <-> server
for x in [2.5, 5.5, 8.5]:
    add_arrow(ax, x + 1.1, y3 + 0.1, 12.4, y3 + 0.1, color=C['blue'])
    add_arrow(ax, 12.4, y3 - 0.1, x + 1.1, y3 - 0.1, color=C['green'])

ax.text(10.5, y3 + 0.35, 'model updates', fontsize=7, color=C['blue'],
        fontstyle='italic', ha='center')
ax.text(10.5, y3 - 0.35, 'global model', fontsize=7, color=C['green'],
        fontstyle='italic', ha='center')

# --- Row 4: Ensemble + SHAP + Output ---
y4 = 1.8
add_box(ax, 3, y4, 2.8, 0.9, 'GA-Optimized\nXGB-RF Ensemble\n(Soft Voting)', C['orange'],
        text_color=C['dark_blue'])
add_box(ax, 7, y4, 2.8, 0.9, 'SHAP\nExplainability\nLayer', C['green'], text_color=C['dark_blue'])
add_box(ax, 11, y4, 2.8, 0.9, 'Fraud / Legitimate\nPrediction\n+ Explanation', C['red'])

# Arrows
add_arrow(ax, 5.5, y3 - 0.4, 3, y4 + 0.45)
add_arrow(ax, 4.4, y4, 5.6, y4)
add_arrow(ax, 8.4, y4, 9.6, y4)

# Metrics annotation
ax.text(11, 0.7, 'Evaluation: AUPRC | F1 | ROC-AUC | Latency < 200ms',
        ha='center', va='center', fontsize=9, color=C['dark_blue'],
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=C['light_grey'],
                  edgecolor=C['grey'], linewidth=1))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'system_architecture.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: system_architecture.png (V31)')

---
## V32 — Federated Learning Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')

ax.text(7, 8.6, 'Federated Learning Architecture', ha='center',
        fontsize=15, fontweight='bold', color=C['dark_blue'])

# --- Central Server ---
server_y = 6.5
server_bg = FancyBboxPatch((4.5, server_y - 0.7), 5, 1.4,
                            boxstyle='round,pad=0.2', facecolor=C['light_blue'],
                            edgecolor=C['blue'], linewidth=2)
ax.add_patch(server_bg)
ax.text(7, server_y + 0.25, 'Flower Server', ha='center', fontsize=12,
        fontweight='bold', color=C['dark_blue'])
ax.text(7, server_y - 0.25, 'FedAvg Aggregation Strategy', ha='center',
        fontsize=9, color=C['blue'])

# --- Round indicator ---
ax.text(12, server_y, 'Communication\nRounds: 10', ha='center',
        fontsize=9, color=C['grey'],
        bbox=dict(boxstyle='round', facecolor=C['light_grey'],
                  edgecolor=C['grey']))

# --- Three Clients ---
client_y = 3.0
client_info = [
    ('Institution A\n(ULB)', '284K txns\n0.17% fraud', C['purple'], 2.5),
    ('Institution B\n(BAF)', '1M txns\n1.10% fraud', C['blue'], 7),
    ('Institution C\n(Synthetic)', '500K txns\n3.59% fraud', C['teal'], 11.5),
]

for name, detail, color, cx in client_info:
    # Client background
    bg = FancyBboxPatch((cx - 1.6, client_y - 1.3), 3.2, 2.6,
                         boxstyle='round,pad=0.2', facecolor='white',
                         edgecolor=color, linewidth=2)
    ax.add_patch(bg)
    ax.text(cx, client_y + 0.8, name, ha='center', fontsize=10,
            fontweight='bold', color=C['dark_blue'])
    ax.text(cx, client_y + 0.2, detail, ha='center', fontsize=8, color=C['grey'])

    # Local models inside client
    add_box(ax, cx - 0.55, client_y - 0.5, 0.9, 0.5, 'XGBoost', color,
            fontsize=7, linewidth=1)
    add_box(ax, cx + 0.55, client_y - 0.5, 0.9, 0.5, 'RF', color,
            fontsize=7, linewidth=1)

    # Local data icon
    ax.text(cx, client_y - 1.05, 'Local Data (Private)', ha='center',
            fontsize=7, color=C['red'], fontstyle='italic')

    # Arrows: client -> server (model updates)
    add_arrow(ax, cx, client_y + 1.3, 7, server_y - 0.7, color=C['orange'], lw=2)
    # Arrows: server -> client (global model)
    offset = 0.3 if cx < 7 else (-0.3 if cx > 7 else 0)
    add_arrow(ax, 7 + offset, server_y - 0.7, cx + offset, client_y + 1.3,
              color=C['green'], lw=1.5, style='->')

# Legend for arrows
ax.annotate('', xy=(1.2, 5.3), xytext=(0.5, 5.3),
            arrowprops=dict(arrowstyle='->', color=C['orange'], lw=2))
ax.text(1.4, 5.3, 'Model updates (no raw data)', fontsize=8, va='center',
        color=C['orange'])
ax.annotate('', xy=(1.2, 4.9), xytext=(0.5, 4.9),
            arrowprops=dict(arrowstyle='->', color=C['green'], lw=1.5))
ax.text(1.4, 4.9, 'Aggregated global model', fontsize=8, va='center',
        color=C['green'])

# Privacy note
ax.text(7, 0.7, 'Raw data never leaves the client — only serialized model parameters are exchanged',
        ha='center', fontsize=9, color=C['red'], fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=C['light_red'],
                  edgecolor=C['red'], linewidth=1, alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fl_architecture.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: fl_architecture.png (V32)')

---
## V33 — Methodology Flowchart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

ax.text(7, 7.6, 'Research Methodology Flowchart', ha='center',
        fontsize=14, fontweight='bold', color=C['dark_blue'])

# Steps as a top-to-bottom flow with two columns for later stages
steps = [
    (3.5, 6.6, 'Data Acquisition\n(3 Datasets: ULB, BAF, Synthetic)', C['blue']),
    (3.5, 5.5, 'Exploratory Data Analysis\n(Class dist, correlations, temporal)', C['blue']),
    (3.5, 4.4, 'Preprocessing\n(Cleaning, Encoding, Scaling, SMOTE)', C['teal']),
    (3.5, 3.3, 'FL Data Partitioning\n(Natural: 1 dataset = 1 client)', C['teal']),
]

# Draw main flow
for x, y, text, color in steps:
    add_box(ax, x, y, 3.8, 0.7, text, color, fontsize=8)

# Arrows between sequential steps
for i in range(len(steps) - 1):
    add_arrow(ax, steps[i][0], steps[i][1] - 0.35,
              steps[i+1][0], steps[i+1][1] + 0.35)

# Branch into centralized and federated
branch_y = 2.2
add_box(ax, 2.5, branch_y, 3.0, 0.7,
        'Centralized Training\n(LR, RF, XGB, Ensemble)', C['orange'],
        text_color=C['dark_blue'], fontsize=8)
add_box(ax, 7, branch_y, 3.0, 0.7,
        'Federated Training\n(Flower FL Simulation)', C['purple'], fontsize=8)

add_arrow(ax, 3.5, 3.3 - 0.35, 2.5, branch_y + 0.35)
add_arrow(ax, 3.5, 3.3 - 0.35, 7, branch_y + 0.35)

# GA optimization feeds into both
add_box(ax, 11, 3.3, 2.5, 0.7, 'GA Hyperparameter\nOptimization', C['orange'],
        text_color=C['dark_blue'], fontsize=8)
add_arrow(ax, 11, 3.3 - 0.35, 7, branch_y + 0.35, color=C['orange'], style='->')
add_arrow(ax, 11, 3.3 - 0.35, 2.5, branch_y + 0.35, color=C['orange'], style='->')

# Converge to evaluation
eval_y = 1.1
add_box(ax, 4.75, eval_y, 3.2, 0.7,
        'Evaluation & Comparison\n(AUPRC, F1, ROC-AUC, Latency)', C['red'], fontsize=8)

add_arrow(ax, 2.5, branch_y - 0.35, 4.75, eval_y + 0.35)
add_arrow(ax, 7, branch_y - 0.35, 4.75, eval_y + 0.35)

# SHAP
add_box(ax, 11, eval_y, 2.5, 0.7, 'SHAP Explainability\n(Global + Local)', C['green'],
        text_color=C['dark_blue'], fontsize=8)
add_arrow(ax, 6.35, eval_y, 9.75, eval_y)

# Phase labels on right
phases = [
    (13.2, 6.6, 'Phase 1', C['blue']),
    (13.2, 5.5, 'Phase 1', C['blue']),
    (13.2, 4.4, 'Phase 2', C['teal']),
    (13.2, 3.3, 'Phase 2–3', C['teal']),
    (13.2, 2.2, 'Phase 3–4', C['purple']),
    (13.2, 1.1, 'Phase 5–6', C['red']),
]
for px, py, label, color in phases:
    ax.text(px, py, label, ha='center', va='center', fontsize=8,
            color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=color, linewidth=1))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'methodology_flowchart.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: methodology_flowchart.png (V33)')

---
## V34 — Gantt Chart (16-Week Project Timeline)

In [ ]:
import pandas as pd

tasks = [
    ('Data Acquisition', 1, 2, C['blue']),
    ('Exploratory Data Analysis', 2, 4, C['blue']),
    ('Preprocessing & SMOTE', 5, 6, C['teal']),
    ('FL Data Partitioning', 5, 6, C['teal']),
    ('Centralized Baselines', 7, 8, C['orange']),
    ('GA Hyperparameter Optimization', 7, 8, C['orange']),
    ('Flower FL Integration', 9, 10, C['purple']),
    ('FL Simulation & Tuning', 10, 12, C['purple']),
    ('SHAP Explainability', 13, 14, C['green']),
    ('Latency Benchmarking', 15, 15, C['red']),
    ('Final Evaluation & Comparison', 15, 16, C['red']),
    ('Report Writing', 8, 16, C['grey']),
    ('Presentation Preparation', 14, 16, C['grey']),
]

fig, ax = plt.subplots(figsize=(14, 7))

y_positions = list(range(len(tasks) - 1, -1, -1))

for i, (name, start, end, color) in enumerate(tasks):
    y = y_positions[i]
    ax.barh(y, end - start + 1, left=start - 0.5, height=0.6,
            color=color, edgecolor='white', linewidth=0.5, alpha=0.85)
    # Label inside bar if wide enough, else to the right
    bar_width = end - start + 1
    if bar_width >= 3:
        ax.text(start + bar_width / 2 - 0.5, y, name, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')
    else:
        ax.text(end + 0.7, y, name, ha='left', va='center',
                fontsize=8, fontweight='bold', color=color)

ax.set_xlabel('Week', fontsize=12)
ax.set_title('Project Timeline — 16-Week Execution Plan', fontsize=14,
             fontweight='bold', color=C['dark_blue'])
ax.set_xticks(range(1, 17))
ax.set_xticklabels([f'W{i}' for i in range(1, 17)])
ax.set_yticks(y_positions)
ax.set_yticklabels([t[0] for t in tasks], fontsize=9)
ax.set_xlim(0.3, 17)
ax.grid(axis='x', alpha=0.3, linestyle='--')

# Phase annotations
phase_spans = [
    (1, 4, 'Phase 1', C['blue']),
    (5, 6, 'Phase 2', C['teal']),
    (7, 8, 'Phase 3', C['orange']),
    (9, 12, 'Phase 4', C['purple']),
    (13, 14, 'Phase 5', C['green']),
    (15, 16, 'Phase 6', C['red']),
]
for start, end, label, color in phase_spans:
    ax.axvspan(start - 0.5, end + 0.5, alpha=0.06, color=color)
    ax.text((start + end) / 2, len(tasks) - 0.3, label, ha='center',
            fontsize=8, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gantt_chart.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: gantt_chart.png (V34)')

---
## V35 — Risk Matrix (Likelihood × Impact)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# 5x5 risk matrix
risk_colors = np.array([
    [1, 1, 2, 2, 3],   # Impact 1 (Low)
    [1, 2, 2, 3, 3],   # Impact 2
    [2, 2, 3, 3, 4],   # Impact 3 (Medium)
    [2, 3, 3, 4, 4],   # Impact 4
    [3, 3, 4, 4, 4],   # Impact 5 (High)
])

color_map = {1: '#2ecc71', 2: '#f1c40f', 3: '#e67e22', 4: '#e74c3c'}
labels_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Critical'}

for i in range(5):
    for j in range(5):
        level = risk_colors[4-i, j]  # flip for display
        rect = plt.Rectangle((j, i), 1, 1, facecolor=color_map[level],
                              edgecolor='white', linewidth=2, alpha=0.7)
        ax.add_patch(rect)

# Place risks on the matrix
risks = [
    (1, 3, 'R1', 'FL convergence\nfailure'),
    (3, 2, 'R2', 'Dataset bias /\nimbalance artifacts'),
    (2, 4, 'R3', 'Latency exceeds\n200ms target'),
    (0, 1, 'R4', 'SHAP computation\ntimeout'),
    (4, 1, 'R5', 'Overfitting to\nsynthetic data'),
    (3, 3, 'R6', 'Hardware /\ncompute limits'),
]

for lx, ly, label, desc in risks:
    ax.plot(lx + 0.5, (4 - ly) + 0.5, 'ko', markersize=20, alpha=0.8)
    ax.text(lx + 0.5, (4 - ly) + 0.5, label, ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

ax.set_xlim(0, 5)
ax.set_ylim(0, 5)
ax.set_xticks([0.5, 1.5, 2.5, 3.5, 4.5])
ax.set_xticklabels(['Very Low', 'Low', 'Medium', 'High', 'Very High'], fontsize=9)
ax.set_yticks([0.5, 1.5, 2.5, 3.5, 4.5])
ax.set_yticklabels(['Very High', 'High', 'Medium', 'Low', 'Very Low'], fontsize=9)
ax.set_xlabel('Likelihood', fontsize=12, fontweight='bold')
ax.set_ylabel('Impact', fontsize=12, fontweight='bold')
ax.set_title('Risk Assessment Matrix', fontsize=14, fontweight='bold',
             color=C['dark_blue'])

# Legend
legend_y = -1.8
for lx, ly, label, desc in risks:
    pass  # We'll put a table below

risk_text = '\n'.join([f'{label}: {desc.replace(chr(10), " ")}'
                        for _, _, label, desc in risks])
ax.text(5.3, 2.5, 'Risk Register:\n\n' + risk_text.replace('\n', '\n'),
        fontsize=8, va='center', fontfamily='monospace',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=C['light_grey'],
                  edgecolor=C['grey']))

# Color legend
for i, (level, color) in enumerate(color_map.items()):
    ax.add_patch(plt.Rectangle((5.3 + i * 1.1, -0.5), 0.4, 0.4,
                                facecolor=color, edgecolor='white', alpha=0.7))
    ax.text(5.8 + i * 1.1, -0.3, labels_map[level], fontsize=7, va='center')

ax.set_xlim(0, 9.5)
ax.set_ylim(-1, 5.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'risk_matrix.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: risk_matrix.png (V35)')

---
## V36 — GDPR / Regulatory Compliance Mapping Table

In [ ]:
import pandas as pd

compliance_data = [
    {
        'Regulation': 'GDPR Article 22',
        'Requirement': 'Right to explanation for automated decisions',
        'System Feature': 'SHAP explanations for each prediction',
        'Compliance Status': 'Addressed',
    },
    {
        'Regulation': 'GDPR Article 5',
        'Requirement': 'Data minimization principle',
        'System Feature': 'Federated Learning — no raw data exchange',
        'Compliance Status': 'Addressed',
    },
    {
        'Regulation': 'EU AI Act (2024)',
        'Requirement': 'Transparency for high-risk AI systems',
        'System Feature': 'SHAP global/local explanations + model documentation',
        'Compliance Status': 'Addressed',
    },
    {
        'Regulation': 'EU AI Act (2024)',
        'Requirement': 'Human oversight capability',
        'System Feature': 'SHAP waterfall/force plots for analyst review',
        'Compliance Status': 'Addressed',
    },
    {
        'Regulation': 'PCI-DSS v4.0',
        'Requirement': 'Protect cardholder data',
        'System Feature': 'FL keeps data at source; PCA-anonymized features (ULB)',
        'Compliance Status': 'Addressed',
    },
    {
        'Regulation': 'Fair Lending (ECOA)',
        'Requirement': 'Non-discriminatory decision-making',
        'System Feature': 'SHAP auditing for feature bias; no demographic features used',
        'Compliance Status': 'Partially Addressed',
    },
    {
        'Regulation': 'Colorado AI Act (2026)',
        'Requirement': 'Impact assessment for high-risk AI',
        'System Feature': 'Ablation study + false positive analysis',
        'Compliance Status': 'Addressed',
    },
]

compliance_df = pd.DataFrame(compliance_data)
compliance_df.to_csv(TABLES_DIR / 'regulatory_compliance_mapping.csv', index=False)

# Render as a figure table
fig, ax = plt.subplots(figsize=(16, 5))
ax.axis('off')
ax.set_title('Regulatory Compliance Mapping', fontsize=14, fontweight='bold',
             color=C['dark_blue'], pad=20)

col_colors = [C['dark_blue']] * 4
cell_colors = []
for _, row in compliance_df.iterrows():
    status = row['Compliance Status']
    if status == 'Addressed':
        cell_colors.append([C['light_green']] * 4)
    else:
        cell_colors.append([C['light_orange']] * 4)

table = ax.table(
    cellText=compliance_df.values,
    colLabels=compliance_df.columns,
    cellColours=cell_colors,
    colColours=[C['light_blue']] * 4,
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.8)

# Bold headers
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold', color=C['dark_blue'])
    cell.set_edgecolor(C['grey'])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'regulatory_compliance_mapping.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: regulatory_compliance_mapping.png + .csv (V36)')

---
## V37 — Soft Voting Ensemble Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')

ax.text(7, 6.6, 'XGBoost–Random Forest Soft Voting Ensemble', ha='center',
        fontsize=14, fontweight='bold', color=C['dark_blue'])

# Input
add_box(ax, 2, 5, 2.5, 0.9, 'Input Transaction\n(30 features)', C['blue'])

# Two models
add_box(ax, 5.5, 5.8, 2.5, 0.7, 'XGBoost\n(Gradient Boosting)', C['orange'],
        text_color=C['dark_blue'], fontsize=9)
add_box(ax, 5.5, 4.2, 2.5, 0.7, 'Random Forest\n(Bagging)', C['green'],
        text_color=C['dark_blue'], fontsize=9)

# Arrows: input -> models
add_arrow(ax, 3.25, 5.3, 4.25, 5.8)
add_arrow(ax, 3.25, 4.7, 4.25, 4.2)

# Probability outputs
add_box(ax, 9, 5.8, 2.0, 0.6, 'P(fraud) = 0.87', C['light_orange'],
        text_color=C['dark_blue'], fontsize=9, fontweight='normal',
        edgecolor=C['orange'])
add_box(ax, 9, 4.2, 2.0, 0.6, 'P(fraud) = 0.91', C['light_green'],
        text_color=C['dark_blue'], fontsize=9, fontweight='normal',
        edgecolor=C['green'])

add_arrow(ax, 6.75, 5.8, 8, 5.8)
add_arrow(ax, 6.75, 4.2, 8, 4.2)

# Averaging
add_box(ax, 9, 3.0, 3.0, 0.8,
        'Soft Voting (Average)\nP = (0.87 + 0.91) / 2', C['purple'], fontsize=9)

add_arrow(ax, 9, 5.5, 9, 3.4, color=C['orange'])
add_arrow(ax, 9, 3.9, 9, 3.4, color=C['green'])

# Final prediction
add_box(ax, 9, 1.6, 2.5, 0.8, 'P(fraud) = 0.89\n> threshold 0.5', C['red'],
        fontsize=10)
add_arrow(ax, 9, 2.6, 9, 2.0)

# Decision
add_box(ax, 9, 0.5, 2.5, 0.6, 'FRAUD DETECTED', C['red'], fontsize=11)
add_arrow(ax, 9, 1.2, 9, 0.8)

# GA optimization annotation
ax.text(2, 2.5, 'Hyperparameters\noptimized via\nGenetic Algorithm',
        ha='center', va='center', fontsize=9, color=C['orange'],
        fontweight='bold', fontstyle='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=C['light_orange'],
                  edgecolor=C['orange'], linewidth=1))
add_arrow(ax, 2.8, 3.0, 4.25, 5.5, color=C['orange'], style='->', lw=1)
add_arrow(ax, 2.8, 2.5, 4.25, 4.2, color=C['orange'], style='->', lw=1)

# SHAP annotation
add_box(ax, 12.5, 1.6, 2.0, 0.8, 'SHAP\nExplanation', C['teal'],
        text_color='white', fontsize=9)
add_arrow(ax, 10.25, 1.6, 11.5, 1.6, color=C['teal'])
ax.text(12.5, 0.5, '"Top drivers:\nV14 = -2.3\nV17 = +1.8"', ha='center',
        fontsize=8, color=C['teal'], fontstyle='italic',
        bbox=dict(boxstyle='round', facecolor=C['light_grey'],
                  edgecolor=C['teal']))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ensemble_diagram.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: ensemble_diagram.png (V37)')

---
## V38 — SMOTE Process Illustration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

np.random.seed(42)

# --- Panel 1: Original Imbalanced Data ---
ax = axes[0]
n_legit = 200
n_fraud = 10

legit_x = np.random.normal(3, 1.5, n_legit)
legit_y = np.random.normal(3, 1.5, n_legit)
fraud_x = np.random.normal(6, 0.6, n_fraud)
fraud_y = np.random.normal(6, 0.6, n_fraud)

ax.scatter(legit_x, legit_y, c=C['blue'], alpha=0.4, s=20, label='Legitimate')
ax.scatter(fraud_x, fraud_y, c=C['red'], alpha=0.9, s=40, marker='X',
           edgecolors='black', linewidths=0.5, label='Fraud')
ax.set_title('1. Original Data\n(Highly Imbalanced)', fontsize=11, fontweight='bold')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=8)
ax.text(0.5, 0.5, f'{n_legit}:{n_fraud}\nratio', ha='center', fontsize=9,
        color=C['red'], fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=C['red']))
ax.set_xlim(-2, 9)
ax.set_ylim(-2, 9)

# --- Panel 2: SMOTE k-NN Interpolation ---
ax = axes[1]
ax.scatter(legit_x, legit_y, c=C['blue'], alpha=0.15, s=15)
ax.scatter(fraud_x, fraud_y, c=C['red'], alpha=0.9, s=50, marker='X',
           edgecolors='black', linewidths=0.5, label='Original Fraud')

# Show k-NN connections for one fraud point
ref_idx = 0
ref_point = (fraud_x[ref_idx], fraud_y[ref_idx])
k = 5
distances = np.sqrt((fraud_x - ref_point[0])**2 + (fraud_y - ref_point[1])**2)
nearest = np.argsort(distances)[1:k+1]

for ni in nearest:
    ax.plot([ref_point[0], fraud_x[ni]], [ref_point[1], fraud_y[ni]],
            'k--', alpha=0.4, linewidth=1)
    # Synthetic point along the line
    lam = np.random.uniform(0.2, 0.8)
    sx = ref_point[0] + lam * (fraud_x[ni] - ref_point[0])
    sy = ref_point[1] + lam * (fraud_y[ni] - ref_point[1])
    ax.scatter(sx, sy, c=C['orange'], s=60, marker='D', edgecolors='black',
              linewidths=0.5, zorder=5)

ax.scatter(ref_point[0], ref_point[1], c=C['red'], s=120, marker='X',
           edgecolors='black', linewidths=1.5, zorder=6)

ax.scatter([], [], c=C['orange'], s=60, marker='D', edgecolors='black',
           label='Synthetic (SMOTE)')
ax.set_title('2. SMOTE k-NN Interpolation\n(k=5 nearest neighbours)', fontsize=11,
             fontweight='bold')
ax.set_xlabel('Feature 1')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(-2, 9)
ax.set_ylim(-2, 9)

# --- Panel 3: Balanced Dataset After SMOTE ---
ax = axes[2]
# Generate synthetic fraud points
n_synthetic = n_legit - n_fraud
synth_fraud_x = np.random.normal(6, 0.9, n_synthetic)
synth_fraud_y = np.random.normal(6, 0.9, n_synthetic)

ax.scatter(legit_x, legit_y, c=C['blue'], alpha=0.4, s=20, label='Legitimate')
ax.scatter(fraud_x, fraud_y, c=C['red'], alpha=0.9, s=40, marker='X',
           edgecolors='black', linewidths=0.5, label='Original Fraud')
ax.scatter(synth_fraud_x, synth_fraud_y, c=C['orange'], alpha=0.6, s=25,
           marker='D', edgecolors='black', linewidths=0.3, label='Synthetic Fraud')

ax.set_title('3. Balanced Dataset\n(After SMOTE)', fontsize=11, fontweight='bold')
ax.set_xlabel('Feature 1')
ax.legend(fontsize=8, loc='upper left')
ax.text(0.5, 0.5, f'{n_legit}:{n_legit}\nbalanced', ha='center', fontsize=9,
        color=C['green'], fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=C['green']))
ax.set_xlim(-2, 9)
ax.set_ylim(-2, 9)

fig.suptitle('SMOTE: Synthetic Minority Over-sampling Technique',
             fontsize=14, fontweight='bold', color=C['dark_blue'], y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'smote_process.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print('Saved: smote_process.png (V38)')

---
## Summary

All Phase 7 visuals generated:

| Visual | File | Description |
|--------|------|-------------|
| V31 | `system_architecture.png` | End-to-end system pipeline |
| V32 | `fl_architecture.png` | FL clients → server → aggregation |
| V33 | `methodology_flowchart.png` | Research methodology flow |
| V34 | `gantt_chart.png` | 16-week project timeline |
| V35 | `risk_matrix.png` | Likelihood × Impact risk assessment |
| V36 | `regulatory_compliance_mapping.png/.csv` | GDPR/EU AI Act/PCI-DSS mapping |
| V37 | `ensemble_diagram.png` | Soft voting ensemble architecture |
| V38 | `smote_process.png` | SMOTE process illustration |